In [8]:
from pathlib import Path
import json
from docx import Document

ROOT = Path("/Users/mariavalencia/Documents/curiosity corner/I love learning/bibliopa")
original = ROOT / "data_reload" / "original"   # the Word docs
prepped  = ROOT / "data_reload" / "prepped"     # the raw json

def count_docx_rows(path):
    """Non-empty rows across all tables in a .docx.
    Entries live in tables, not paragraphs, so counting paragraphs returns 1."""
    doc = Document(path)
    rows = 0
    for table in doc.tables:
        for row in table.rows:
            if any(cell.text.strip() for cell in row.cells):
                rows += 1
    return rows

In [9]:
# --- Word documents: non-empty table rows ---
docx_counts = {}
for f in sorted(original.glob("*.docx")):
    if f.name.startswith("~$"):   # skip Word lock/temp files
        continue
    docx_counts[f.name] = count_docx_rows(f)

for name, n in docx_counts.items():
    print(f"{n:>6}  {name}")

print(f"\nfiles: {len(docx_counts)}")
print(f"TOTAL docx rows: {sum(docx_counts.values())}")

   215  ANTIKE TEXTE.docx
    35  Autographen.docx
   227  BIBLIOPHILE BÜCHER.docx
   654  BIOGRAPHIE.docx
   371  BRIEFE TAGEBÜCHER WERKE.docx
   195  BUCH VERLAG ALMANACHE.docx
    78  BUDDHISMUS, HINDUISMUS.docx
   138  CHRISTENTUM.docx
   238  DEUTSCHE LITERATUR MONOGRAPHIEN.docx
   483  DEUTSCHE LITERATUR TEXTE.docx
   106  DEUTSCHLAND.docx
   665  ERSTAUSGABEN A - G.docx
   673  ERSTAUSGABEN H - M.docx
   616  ERSTAUSGABEN N - Z.docx
    65  ESOTERIK.docx
   307  ETHNOLOGIE PSYCHOLOGIE NATURWISSENSCHAFT.docx
   950  FREMDSPRACHIGE LITERATUR IN ÜBERSETZUNGEN.docx
   111  FRÜHES CHRISTENTUM.docx
    58  GESCHICHTE ALLGEMEIN LÄNDER.docx
   249  Griechenland Rom.docx
   673  INSEL-BÜCHEREI.docx
    68  ISLAM.docx
   337  Illustrierte Bücher.docx
   224  JUDENTUM.docx
   940  KUNSTBÜCHER.docx
    98  Kinder- und Jugendliteratur.docx
   287  LITERATUR-UND SPRACHWISSENSCHAFT.docx
   153  MITTELALTER.docx
    69  MONOGRAPHIEN FREMDSPRACHIGER AUTOREN.docx
   273  MUSIK UND THEATER.docx
  

In [10]:
# --- Prepped JSON: number of records per file ---
json_counts = {}
for f in sorted(prepped.glob("*.json")):
    with open(f, encoding="utf-8") as fh:
        records = json.load(fh)
    json_counts[f.name] = len(records)

for name, n in json_counts.items():
    print(f"{n:>6}  {name}")

print(f"\nfiles: {len(json_counts)}")
print(f"TOTAL json records: {sum(json_counts.values())}")

    73  aegypten.json
   213  antike.json
    35  autographen.json
   227  bibliophile.json
   655  biographie.json
   371  briefe.json
   195  buch.json
    72  buddhismus.json
   134  christentum.json
   238  de-lit-monographien.json
   479  de-lit-texte.json
    97  deutschland.json
   665  erstausgaben1.json
   673  erstausgaben2.json
   611  erstausgaben3.json
    61  esoterik.json
   305  ethnologie.json
   950  fremdsprachige.json
    97  fruehes.json
    57  geschichte.json
   244  griechenland.json
   337  illustrierte.json
   673  insel.json
    66  islam.json
   220  judentum.json
    98  kinder.json
   930  kunstbuecher.json
   287  literatur.json
    54  maerchen.json
   151  mittelalter.json
    69  monographien.json
   271  musik.json
    58  mythologie.json
    98  neuzeit.json
   307  oesterreich.json
   179  okkultismus.json
   595  philosophie.json
   311  reisen.json
    89  religion.json
    89  renaissance.json
   409  russland.json
   240  signierte.json
    79  

In [11]:
# --- Side-by-side CSV: docx rows vs json records per category ---
import csv

def key_of(name):
    name = name.upper()
    # the 3 erstausgaben docs are split A-G / H-M / N-Z but share one json topic
    return "ERSTAUSGABEN (combined)" if name.startswith("ERSTAUSGABEN") else name

# docx side, aggregated by category key
docx = {}
for f in sorted(original.glob("*.docx")):
    if f.name.startswith("~$"):
        continue
    docx[key_of(f.stem)] = docx.get(key_of(f.stem), 0) + count_docx_rows(f)

# json side, joined on the 'topic' field stored inside each record
jsn = {}
for f in sorted(prepped.glob("*.json")):
    recs = json.load(open(f, encoding="utf-8"))
    topic = recs[0]["topic"] if recs else f.stem
    jsn[key_of(topic)] = jsn.get(key_of(topic), 0) + len(recs)

# utf-8-sig writes a BOM so Excel reads the German umlauts correctly
out_csv = ROOT / "data_reload" / "book_counts_comparison.csv"
with open(out_csv, "w", newline="", encoding="utf-8-sig") as f:
    w = csv.writer(f)
    w.writerow(["category", "docx_rows", "json_records", "difference"])
    for key in sorted(set(docx) | set(jsn)):
        d, j = docx.get(key, 0), jsn.get(key, 0)
        w.writerow([key, d, j, d - j])
    w.writerow(["TOTAL", sum(docx.values()), sum(jsn.values()),
                sum(docx.values()) - sum(jsn.values())])

print(f"wrote {out_csv}")

wrote /Users/mariavalencia/Documents/curiosity corner/I love learning/bibliopa/data_reload/book_counts_comparison.csv
